**Import Key Libraries**

In [25]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import lxml
import pandas as pd
import re
from datetime import datetime
from selenium.webdriver.common.keys import Keys 

In [26]:
import json
import platform
import zipfile
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit
from urllib.request import urlopen, urlretrieve

from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

PRODUCT_LINK_SELECTOR = 'a[href*="/p/"][href*="pid="]'
PRODUCT_SCHEMA_SELECTOR = 'script[type="application/ld+json"]'


def create_chrome_driver() -> webdriver.Chrome:
    """Create Chrome WebDriver, including a Windows-on-ARM workaround."""
    if platform.system() != "Windows" or platform.machine().lower() not in {"arm64", "aarch64"}:
        return webdriver.Chrome()

    chrome_path = Path(r"C:\Program Files\Google\Chrome\Application\chrome.exe")
    if not chrome_path.is_file():
        raise FileNotFoundError(f"Chrome was not found at {chrome_path}")

    version_directories = [
        directory.name
        for directory in chrome_path.parent.iterdir()
        if directory.is_dir() and directory.name[0:1].isdigit()
    ]
    if not version_directories:
        raise RuntimeError("Could not determine the installed Chrome version.")

    chrome_version = max(version_directories, key=lambda value: tuple(map(int, value.split("."))))
    chrome_build = ".".join(chrome_version.split(".")[:3])
    metadata_url = (
        "https://googlechromelabs.github.io/chrome-for-testing/"
        "latest-patch-versions-per-build-with-downloads.json"
    )

    with urlopen(metadata_url, timeout=30) as response:
        metadata = json.load(response)

    build_metadata = metadata["builds"].get(chrome_build)
    if build_metadata is None:
        raise RuntimeError(f"No ChromeDriver release exists for Chrome build {chrome_build}.")

    driver_download = next(
        download
        for download in build_metadata["downloads"]["chromedriver"]
        if download["platform"] == "win64"
    )
    driver_version = build_metadata["version"]
    driver_directory = Path.home() / ".cache" / "selenium" / "chromedriver" / driver_version
    driver_path = driver_directory / "chromedriver-win64" / "chromedriver.exe"

    if not driver_path.is_file():
        driver_directory.mkdir(parents=True, exist_ok=True)
        archive_path = driver_directory / "chromedriver-win64.zip"
        urlretrieve(driver_download["url"], archive_path)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(driver_directory)
        archive_path.unlink()

    options = Options()
    options.binary_location = str(chrome_path)
    return webdriver.Chrome(service=Service(str(driver_path)), options=options)


def build_page_url(search_url: str, page_number: int) -> str:
    """Return a search URL with an explicit Flipkart page number."""
    parts = urlsplit(search_url)
    query = dict(parse_qsl(parts.query, keep_blank_values=True))
    query["page"] = str(page_number)
    return urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(query), parts.fragment))


def canonicalize_product_url(product_url: str) -> str:
    """Remove volatile tracking parameters while retaining the product identity."""
    parts = urlsplit(product_url)
    query = dict(parse_qsl(parts.query, keep_blank_values=True))
    stable_query = {key: query[key] for key in ("pid", "marketplace") if key in query}
    return urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(stable_query), ""))


def find_product_schema(value):
    """Recursively find a schema.org Product object in decoded JSON-LD."""
    if isinstance(value, dict):
        schema_type = value.get("@type")
        if schema_type == "Product" or (isinstance(schema_type, list) and "Product" in schema_type):
            return value
        for child in value.values():
            product = find_product_schema(child)
            if product is not None:
                return product
    elif isinstance(value, list):
        for child in value:
            product = find_product_schema(child)
            if product is not None:
                return product
    return None


def read_product_schema(driver: webdriver.Chrome) -> dict:
    """Read stable schema.org product data instead of volatile CSS classes."""
    scripts = WebDriverWait(driver, 30).until(
        lambda current_driver: current_driver.find_elements(By.CSS_SELECTOR, PRODUCT_SCHEMA_SELECTOR)
    )
    for script in scripts:
        raw_json = script.get_attribute("textContent")
        if not raw_json:
            continue
        try:
            product = find_product_schema(json.loads(raw_json))
        except json.JSONDecodeError:
            continue
        if product is not None:
            return product
    raise ValueError("No schema.org Product data was found on the page.")

### Step1: Get all product Links

In [ ]:
# Inputs to search
search_box_text = "sports shoes for women"
website_link = "https://www.flipkart.com/"

session_start_time = datetime.now().time()
print(f"Session Start Time: {session_start_time} ---------------------------> ")

driver = create_chrome_driver()
try:
    driver.get(website_link)
    driver.maximize_window()

    print("Waiting for search input...")
    search_input = WebDriverWait(driver, 120).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, '[autocomplete="off"]'))
    )

    print("Typing in search input...")
    search_input.send_keys(search_box_text)

    print("Submitting search form...")
    search_input.send_keys(Keys.RETURN)

    print("Waiting for search results...")
    WebDriverWait(driver, 120).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, PRODUCT_LINK_SELECTOR))
    )

    print("Collecting pagination links...")
    all_pagination_links = [build_page_url(driver.current_url, page) for page in range(1, 26)]
    print("Pagination Links Count:", len(all_pagination_links))

    print("Collecting Product Detail Page Links")
    all_product_links = set()

    for page_number, link in enumerate(all_pagination_links, start=1):
        driver.get(link)
        WebDriverWait(driver, 120).until(
            lambda current_driver: current_driver.execute_script("return document.readyState") == "complete"
        )

        try:
            all_products = WebDriverWait(driver, 30).until(
                lambda current_driver: current_driver.find_elements(
                    By.CSS_SELECTOR,
                    PRODUCT_LINK_SELECTOR,
                )
            )
        except TimeoutException:
            print(f"Page {page_number} contained no product links; skipping {link}")
            continue

        page_product_links = {
            canonicalize_product_url(element.get_attribute("href"))
            for element in all_products
            if element.get_attribute("href")
        }
        all_product_links.update(page_product_links)
        print(f"Page {page_number}: captured {len(page_product_links)} unique links")

    print("Total unique Product Detail Page Links:", len(all_product_links))

    df_product_links = pd.DataFrame(sorted(all_product_links), columns=["product_links"])
    df_product_links.to_csv("flipkart_product_links.csv", index=False)
finally:
    driver.quit()

session_end_time = datetime.now().time()
print(f"Session End Time: {session_end_time} ---------------------------> ")

Session Start Time: 21:47:54.309073 ---------------------------> 
Waiting for search input...
Typing in search input...
Submitting search form...
Waiting for search results...
Pagination Links Count: 25
All Pagination Links: ['https://www.flipkart.com/search?q=sports+shoes+for+women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=1', 'https://www.flipkart.com/search?q=sports+shoes+for+women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=2', 'https://www.flipkart.com/search?q=sports+shoes+for+women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=3', 'https://www.flipkart.com/search?q=sports+shoes+for+women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=4', 'https://www.flipkart.com/search?q=sports+shoes+for+women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=5', 'https://www.flipkart.com/search?q=sports+shoes+for+women&otracker=search&

In [23]:
# Verify ChromeDriver and the stable Flipkart product-link selector.
test_driver = create_chrome_driver()
try:
    test_url = "https://www.flipkart.com/search?q=sports+shoes+for+women&page=1"
    test_driver.get(test_url)
    product_links = WebDriverWait(test_driver, 30).until(
        lambda current_driver: current_driver.find_elements(
            By.CSS_SELECTOR,
            PRODUCT_LINK_SELECTOR,
        )
    )
    print("Chrome WebDriver is ready:", test_driver.capabilities["browserVersion"])
    print("Product links found:", len(product_links))
    print("Page 2 URL:", build_page_url(test_driver.current_url, 2))
finally:
    test_driver.quit()

Chrome WebDriver is ready: 151.0.7922.175
Product links found: 125
Page 2 URL: https://www.flipkart.com/search?q=sports+shoes+for+women&page=2


### Step2: Get Individual product information

In [28]:
session_start_time = datetime.now().time()
print(f"Session Start Time: {session_start_time} ---------------------------> ")

# Read and normalize the CSV file containing product links.
df_product_links = pd.read_csv("flipkart_product_links.csv")
df_product_links["product_links"] = df_product_links["product_links"].map(
    canonicalize_product_url
)
df_product_links = df_product_links.drop_duplicates(subset=["product_links"])

# Remove this line to scrape all products. The sample run processes only 10 products.
df_product_links = df_product_links.head(10)
all_product_links = df_product_links["product_links"].tolist()
print("Collecting Individual Product Detail Information")

complete_product_details = []
unavailable_products = []
failed_products = []

driver = create_chrome_driver()
try:
    for index, product_page_link in enumerate(all_product_links, start=1):
        try:
            driver.get(product_page_link)
            WebDriverWait(driver, 120).until(
                lambda current_driver: current_driver.execute_script(
                    "return document.readyState"
                ) == "complete"
            )

            product = read_product_schema(driver)
            offer = product.get("offers", {})
            if isinstance(offer, list):
                offer = offer[0] if offer else {}

            availability = str(offer.get("availability", ""))
            if availability and not availability.endswith("/InStock"):
                unavailable_products.append(product_page_link)
                status = availability.rsplit("/", 1)[-1]
                print(f"URL {index} is unavailable: {status}")
                continue

            brand_data = product.get("brand", {})
            if isinstance(brand_data, dict):
                brand = brand_data.get("name", "")
            else:
                brand = str(brand_data)

            title = product.get("name", "")
            price = offer.get("price", "")

            rating_data = product.get("aggregateRating", {})
            avg_rating = rating_data.get("ratingValue", "")
            total_ratings = rating_data.get("ratingCount", "")

            # JSON-LD does not consistently expose list price, so discount is left blank.
            discount = ""
            complete_product_details.append(
                [
                    product_page_link,
                    title,
                    brand,
                    price,
                    discount,
                    avg_rating,
                    total_ratings,
                ]
            )
            print(f"URL {index} completed")
        except Exception as error:
            failed_products.append(
                {"link": product_page_link, "error": str(error)}
            )
            print(f"URL {index} failed to parse: {error}")
finally:
    driver.quit()

columns = [
    "product_link",
    "title",
    "brand",
    "price",
    "discount",
    "avg_rating",
    "total_ratings",
]
df = pd.DataFrame(complete_product_details, columns=columns)

# A canonical link uniquely identifies a Flipkart product.
duplicate_subset = ["product_link"]
df_duplicate_products = df[df.duplicated(subset=duplicate_subset)]
df = df.drop_duplicates(subset=duplicate_subset)

df_unavailable_products = pd.DataFrame(unavailable_products, columns=["link"])
df_failed_products = pd.DataFrame(failed_products, columns=["link", "error"])

print("Total product pages processed:", len(all_product_links))
print("Final Total Products:", len(df))
print("Total Unavailable Products:", len(df_unavailable_products))
print("Total Failed Products:", len(df_failed_products))
print("Total Duplicate Products:", len(df_duplicate_products))

df.to_csv("flipkart_product_data.csv", index=False)
df_unavailable_products.to_csv("unavailable_products.csv", index=False)
df_failed_products.to_csv("failed_products.csv", index=False)
df_duplicate_products.to_csv("duplicate_products.csv", index=False)

session_end_time = datetime.now().time()
print(f"Session End Time: {session_end_time} ---------------------------> ")

Session Start Time: 21:53:27.437787 ---------------------------> 
URL 1 completed
URL 2 completed
URL 3 completed
URL 4 completed
URL 5 completed
URL 6 completed
URL 7 completed
URL 8 completed
URL 9 completed
URL 10 completed
Total product pages processed: 10
Final Total Products: 10
Total Unavailable Products: 0
Total Failed Products: 0
Total Duplicate Products: 0
Session End Time: 21:53:41.815550 ---------------------------> 
